# SQL query from table names - Continued

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

## The old Prompt

In [2]:
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table (motorcycle riders):
{
  "tableName": "riders",
  "fields": [
    {
      "name": "rider_id",
      "type": "int"
    },
    {
      "name": "name",
      "type": "varchar"
    },
    {
      "name": "nationality",
      "type": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table (teams):
{
  "tableName": "teams",
  "fields": [
    {
      "name": "team_id",
      "type": "int"
    },
    {
      "name": "name",
      "type": "varchar"
    },
    {
      "name": "country",
      "type": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table (circuits):
{
  "tableName": "circuits",
  "fields": [
    {
      "name": "circuit_id",
      "type": "int"
    },
    {
      "name": "name",
      "type": "varchar"
    },
    {
      "name": "location",
      "type": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
fourth table (races):
{
  "tableName": "races",
  "fields": [
    {
      "name": "race_id",
      "type": "int"
    },
    {
      "name": "rider_id",
      "type": "int"
    },
    {
      "name": "team_id",
      "type": "int"
    },
    {
      "name": "circuit_id",
      "type": "int"
    },
    {
      "name": "year",
      "type": "int"
    },
    {
      "name": "championships",
      "type": "int"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [3]:
context = [ {'role':'system', 'content':"""
          CREATE TABLE riders (
              rider_id INTEGER PRIMARY KEY,
              name TEXT,
              nationality TEXT
          );

          CREATE TABLE teams (
              team_id INTEGER PRIMARY KEY,
              name TEXT,
              country TEXT
          );

          CREATE TABLE circuits (
              circuit_id INTEGER PRIMARY KEY,
              name TEXT,
              location TEXT
          );

          CREATE TABLE races (
              race_id INTEGER PRIMARY KEY,
              rider_id INTEGER,
              team_id INTEGER,
              circuit_id INTEGER,
              year INTEGER,
              championships INTEGER,
              FOREIGN KEY (rider_id) REFERENCES riders(rider_id),
              FOREIGN KEY (team_id) REFERENCES teams(team_id),
              FOREIGN KEY (circuit_id) REFERENCES circuits(circuit_id)
          );
             
              
                -- Riders
                INSERT INTO riders VALUES (1, 'Marc Márquez', 'Spanish');
                INSERT INTO riders VALUES (2, 'Valentino Rossi', 'Italian');
                INSERT INTO riders VALUES (3, 'Jorge Lorenzo', 'Spanish');

                -- Teams
                INSERT INTO teams VALUES (1, 'Repsol Honda', 'Japan');
                INSERT INTO teams VALUES (2, 'Yamaha Factory', 'Japan');
                INSERT INTO teams VALUES (3, 'Ducati Lenovo', 'Italy');

                -- Circuits
                INSERT INTO circuits VALUES (1, 'Silverstone', 'UK');
                INSERT INTO circuits VALUES (2, 'Circuito de Jerez', 'Spain');
                INSERT INTO circuits VALUES (3, 'Mugello', 'Italy');

                -- Races
                INSERT INTO races VALUES (1, 1, 1, 1, 2019, 8);
                INSERT INTO races VALUES (2, 2, 2, 2, 2009, 7);
                INSERT INTO races VALUES (3, 3, 2, 3, 2015, 5);

"""} ]



In [4]:
#FEW SHOT SAMPLES
context.append( {'role':'system', 'content':"""
                -- Maintain the SQL order simple and efficient as you can,
                -- using valid SQL Lite, answer the following questions for the table provided above.
                 
                Here are some examples of questions and the correct SQL:

                Question: Which driver has the most championships?
                SQL: SELECT name, championships FROM riders 
                     JOIN races ON riders.rider_id = races.rider_id 
                     ORDER BY championships DESC LIMIT 1;

                Question: How many Spanish pilots are there?
                SQL: SELECT COUNT(*) FROM riders 
                     WHERE nationality = 'Spanish';

                Question: Which team has won the most races?
                SQL: SELECT teams.name, COUNT(*) as wins FROM races 
                     JOIN teams ON races.team_id = teams.team_id 
                     GROUP BY teams.name 
                     ORDER BY wins DESC LIMIT 1;

                Now answer the following question following the same format. 
                Start your answer with "\ this is your SQL" and then the SQL query
                in white letters with black background, followed by a brief explanation.
"""
})

In [5]:
#Function to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
    )

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [6]:
#new
context_user = context.copy()
print(return_CCRMSQL("""Which driver is the best in terms of championships and races won?""", context_user))

\ this is your SQL
```sql
SELECT riders.name, SUM(races.championships) as total_championships, COUNT(races.race_id) as total_wins
FROM riders
JOIN races ON riders.rider_id = races.rider_id
GROUP BY riders.name
ORDER BY total_championships DESC, total_wins DESC
LIMIT 1;
```

Explanation: This SQL query joins the `riders` and `races` tables on the rider_id, calculates the total championships won and total races won for each rider, groups the results by rider name, and then orders the results by total championships in descending order and total wins in descending order. Finally, it limits the output to only the top result, which gives us the rider who is the best in terms of championships and races won.


In [7]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("Which driver is the best in terms of championships and races won?", old_context_user))

This is your SQL:
```sql
SELECT r.name AS rider_name, COUNT(ra.race_id) AS total_races, SUM(ra.championships) AS total_championships
FROM riders r
JOIN races ra ON r.rider_id = ra.rider_id
GROUP BY r.name
ORDER BY total_championships DESC, total_races DESC
LIMIT 1;
```

This SQL query selects the rider with the most championships and races won by joining the "riders" and "races" tables, counting the total number of races and summing the total championships for each rider. The results are then grouped by rider, ordered by total championships and total races in descending order, and the top result is limited to 1 to show the best driver.


In [8]:
#new
print(return_CCRMSQL("How many world championships does each country have?", context_user))

\ this is your SQL
```sql
SELECT nationality, SUM(championships) as total_championships
FROM riders
JOIN races ON riders.rider_id = races.rider_id
GROUP BY nationality;
```

Explanation: This SQL query joins the `riders` and `races` tables on the `rider_id` to calculate the total number of world championships won by riders from each country. The query then groups the results by nationality and calculates the sum of championships for each country.


In [9]:
#old
print(return_CCRMSQL("How many world championships does each country have?", old_context_user))

This is your SQL:
```sql
SELECT teams.country, SUM(races.championships) AS total_championships
FROM teams
JOIN races ON teams.team_id = races.team_id
GROUP BY teams.country;
```

This SQL query joins the "teams" and "races" tables on the team_id, then groups the results by country. It calculates the total number of championships won by each country by summing up the championships from the races table.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

# 📊 Technical Report: Comparative Evaluation of Text-to-SQL Prompting

## 1. Summary of Findings
This report analyzes the performance shift between a standard system prompt (**Old Prompt**) and a structured approach based on the Ohio State University study (**New Prompt**). The core difference lies in the transition from descriptive JSON metadata to a combination of **DDL (Data Definition Language)**, **Data Shadowing**, and **Few-Shot Learning**.

* **Old Prompt:** Relied on high-level instructions and JSON-formatted schema. While it achieved basic joins, the logic was sometimes inconsistent when handling ambiguous terms.
* **New Prompt:** By utilizing `CREATE TABLE` statements and `INSERT INTO` samples, the model gained a grounded understanding of the data. The inclusion of few-shot examples acted as a structural template, forcing the output to match a specific professional format.

---

## 2. Variations & Hallucinations Analysis
During testing, several key logical variations and potential hallucinations were identified:

* **Semantic Mapping Conflicts:** * In the query *"How many world championships does each country have?"*, the **Old Prompt** joined the `teams` table (using team country), while the **New Prompt** joined the `riders` table (using rider nationality). 
    * **Finding:** Without explicit constraints, GPT alternates between these logic paths. The New Prompt provided more "grounded" results because the sample data explicitly linked names like "Marc Márquez" to "Spanish" nationality.
* **Structural Efficiency:**
    * The New Prompt produced cleaner, more standard SQL. The DDL acts as a "guardrail," preventing the model from hallucinating columns that do not exist in the physical schema.
* **Formatting Consistency:**
    * The Old Prompt sometimes ignored formatting instructions (like background colors or prefixes). The New Prompt, thanks to **Few-Shot Samples**, followed the `\ this is your SQL` prefix and explanation format with 100% consistency.


---

## 3. Lessons Learned
1.  **DDL is the Gold Standard:** LLMs are pre-trained on massive code repositories. Providing the schema as `CREATE TABLE` blocks allows the model to leverage its native understanding of SQL constraints and relationships.
2.  **The Power of Few-Shot Samples:** Providing 3-4 examples of "Question -> SQL" pairs is the single most effective way to control both the logic and the output style of the model.
3.  **Data Shadowing (In-Context Samples):** Including a few `INSERT` statements allows the model to perform **Entity Linking** correctly. It understands that 'Silverstone' is a `name` in the `circuits` table, not just a random string.
4.  **Deterministic Control:** Maintaining `temperature=0` is non-negotiable for technical tasks to ensure that the generated SQL is reproducible and logical rather than "creative."

---

## 4. Final Conclusion
The "New Prompt" methodology